# Predicción de Gestos LSM con Ensamble (BiLSTM + GRU)


## 1. Importar Librerías y Definir Clases

In [1]:
import os
import cv2
import mediapipe as mp
import numpy as np
import joblib
import tensorflow as tf
from pathlib import Path
from tensorflow.keras.models import load_model
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from tqdm import tqdm
import tkinter as tk
from tkinter import filedialog
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='absl')

2025-10-26 23:50:11.645666: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-26 23:50:11.657283: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761544211.670142  175944 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761544211.674163  175944 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1761544211.684044  175944 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
class FeatureExtractor:
    def __init__(self):
        self.feature_vector_size = 21 * 3 * 2
        self._hands = None

    @property
    def hands(self):
        if self._hands is None:
            print("Inicializando modelo MediaPipe Hands...")
            mp_hands = mp.solutions.hands
            self._hands = mp_hands.Hands(
                static_image_mode=False,
                max_num_hands=2,
                min_detection_confidence=0.5,
                min_tracking_confidence=0.5
            )
        return self._hands 

    def __getstate__(self):
        state = self.__dict__.copy()
        state['_hands'] = None
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        self._hands = None 

    def extract_hand_keypoints(self, video_path):
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"Error: No se pudo abrir el video {video_path}")
            return None

        video_keypoints = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = self.hands.process(image_rgb) 
            frame_keypoints = np.zeros(self.feature_vector_size)

            if results.multi_hand_landmarks:
                for i, hand_landmarks in enumerate(results.multi_hand_landmarks):
                    if i >= 2: continue
                    landmarks = np.array(
                        [[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark]
                    ).flatten()
                    start_index = i * (21 * 3)
                    end_index = start_index + len(landmarks)
                    frame_keypoints[start_index:end_index] = landmarks
            video_keypoints.append(frame_keypoints)
        cap.release()
        return np.array(video_keypoints)

    def calculate_kinematic_features(self, keypoints_sequence):
        if keypoints_sequence.shape[0] < 2:
            return np.zeros_like(keypoints_sequence)
        velocities = np.diff(keypoints_sequence, axis=0)
        velocities = np.vstack([np.zeros(velocities.shape[1]), velocities])
        return velocities

    def process_video(self, video_path):
        keypoints = self.extract_hand_keypoints(video_path)
        if keypoints is None or keypoints.shape[0] == 0:
            return None
        velocities = self.calculate_kinematic_features(keypoints)
        combined_features = np.hstack([keypoints, velocities])
        return combined_features

class VideoFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.extractor = None 

    def _get_extractor(self):
        if self.extractor is None:
            self.extractor = FeatureExtractor()
        return self.extractor

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        all_sequences = []
        print("Paso 1/4: Extrayendo características del video...")
        extractor = self._get_extractor() 
        
        for video_path in tqdm(X, desc="Extrayendo secuencia"):
            features = extractor.process_video(video_path) 
            if features is not None and features.shape[0] > 0:
                all_sequences.append(features)
        return all_sequences

    def __getstate__(self):
        state = self.__dict__.copy()
        state['extractor'] = None
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        self.extractor = None 

class SequencePadder(BaseEstimator, TransformerMixin):
    def __init__(self, percentile=95, target_frames_=0):
        self.percentile = percentile
        self.target_frames_ = target_frames_

    def fit(self, X, y=None):
        if self.target_frames_ == 0:
            print("Advertencia: target_frames_ no se cargó, ajustando a los datos actuales.")
            sequence_lengths = [seq.shape[0] for seq in X]
            if sequence_lengths:
                self.target_frames_ = int(np.percentile(sequence_lengths, self.percentile))
        print(f"Longitud de secuencia objetivo: {self.target_frames_} fotogramas.")
        return self

    def transform(self, X, y=None):
        print("Paso 2/4: Estandarizando longitud de secuencia...")
        processed_sequences = []
        for seq in tqdm(X, desc="Padding"):
            num_frames, num_features = seq.shape
            if num_frames > self.target_frames_:
                processed_seq = seq[:self.target_frames_, :]
            elif num_frames < self.target_frames_:
                pad_width = self.target_frames_ - num_frames
                padding = np.zeros((pad_width, num_features))
                processed_seq = np.vstack([seq, padding])
            else:
                processed_seq = seq
            processed_sequences.append(processed_seq)
        return np.array(processed_sequences)

class TemporalFeatureProcessor(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=0.95, feature_pipeline_=None):
        self.n_components = n_components
        self.feature_pipeline_ = feature_pipeline_

    def fit(self, X, y=None):
        if self.feature_pipeline_ is None:
            print("Advertencia: feature_pipeline_ no se cargó, ajustando a los datos actuales.")
            self.feature_pipeline_ = Pipeline([\
                ('scaler', MinMaxScaler()),
                ('pca', PCA(n_components=self.n_components))
            ])
            n_videos, n_frames, n_features = X.shape
            reshaped_data = X.reshape(-1, n_features)
            self.feature_pipeline_.fit(reshaped_data)
        print("Scaler y PCA listos.")
        return self

    def transform(self, X, y=None):
        print("Paso 3/4: Aplicando Scaler y PCA...")
        n_videos, n_frames, n_features = X.shape
        reshaped_data = X.reshape(-1, n_features)
        transformed_data = self.feature_pipeline_.transform(reshaped_data)
        n_transformed_features = transformed_data.shape[1]
        final_data = transformed_data.reshape(n_videos, n_frames, n_transformed_features)
        print(f"Características reducidas a {n_transformed_features} componentes.")
        return final_data

print("Librerías y clases definidas.")

Librerías y clases definidas.


## 2. Indicar Rutas de Archivos

Define las rutas a tus artefactos. Asumimos que los modelos están en una carpeta `models` y el pipeline/encoder en `processed_data_pipeline`.

In [3]:
notebook_dir = Path.cwd() 
parent_dir = notebook_dir

model_path = parent_dir / "models" / "model_7_bilstm_2025-10-26_21-37-27.h5"
model_gru_path = parent_dir / "models" / "model_2_gru_2025-10-26_20-31-31.h5"
pipeline_path = parent_dir / "processed_data_pipeline" / "pipeline_procesamiento.joblib"
encoder_path = parent_dir / "processed_data_pipeline" / "label_encoder.joblib"

print(f"Ruta Modelo BiLSTM: {model_path}")
print(f"Ruta Modelo GRU: {model_gru_path}")
print(f"Ruta Pipeline:   {pipeline_path}")
print(f"Ruta Encoder:    {encoder_path}")

# --- Seleccionar Carpeta de Videos ---
root = tk.Tk()
root.withdraw()

print("\nAbriendo explorador para seleccionar la CARPETA de videos...")
folder_path_str = filedialog.askdirectory(
    title="Selecciona la carpeta con los videos a predecir"
)

video_files_list = []
if folder_path_str:
    folder_path = Path(folder_path_str)
    print(f"Carpeta seleccionada: {folder_path_str}")
    
    video_extensions = ["*.mp4", "*.avi", "*.mov"]
    video_path_generators = [folder_path.glob(ext) for ext in video_extensions]
    
    all_video_paths = [str(p) for gen in video_path_generators for p in gen]
    
    video_files_list = sorted(all_video_paths)
    print(f"Se encontraron {len(video_files_list)} videos.")
else:
    print("No se seleccionó ninguna carpeta.")

root.destroy()


Ruta Modelo BiLSTM: /mnt/c/Users/Raven/Desktop/ProyectoIntegrador/github/TC5035-PI-E51/Avance5/models/model_7_bilstm_2025-10-26_21-37-27.h5
Ruta Modelo GRU: /mnt/c/Users/Raven/Desktop/ProyectoIntegrador/github/TC5035-PI-E51/Avance5/models/model_2_gru_2025-10-26_20-31-31.h5
Ruta Pipeline:   /mnt/c/Users/Raven/Desktop/ProyectoIntegrador/github/TC5035-PI-E51/Avance5/processed_data_pipeline/pipeline_procesamiento.joblib
Ruta Encoder:    /mnt/c/Users/Raven/Desktop/ProyectoIntegrador/github/TC5035-PI-E51/Avance5/processed_data_pipeline/label_encoder.joblib

Abriendo explorador para seleccionar la CARPETA de videos...
Carpeta seleccionada: /mnt/c/Users/Raven/Desktop/ProyectoIntegrador/test
Se encontraron 8 videos.


## 3. Cargar Artefactos (Modelos, Pipeline, Codificador)

In [4]:
print(f"Cargando modelo BiLSTM desde {model_path}...")
model_gru = load_model(model_path)
print("Modelo BiLSTM cargado.")

print(f"Cargando modelo GRU desde {model_gru_path}...")
model_mlp = load_model(model_gru_path)
print("Modelo GRU cargado.")

print(f"Cargando pipeline desde {pipeline_path}...")
pipeline = joblib.load(pipeline_path)
print("Pipeline cargado.")

print(f"Cargando codificador desde {encoder_path}...")
encoder = joblib.load(encoder_path)
print("Codificador cargado.")

print("\n--- Resumen del Modelo BiLSTM (Recurrente) ---")
model_gru.summary()

print("\n--- Resumen del Modelo GRU (Plano) ---")
model_mlp.summary()

Cargando modelo BiLSTM desde /mnt/c/Users/Raven/Desktop/ProyectoIntegrador/github/TC5035-PI-E51/Avance5/models/model_7_bilstm_2025-10-26_21-37-27.h5...


I0000 00:00:1761544223.307060  175944 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21458 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


Modelo BiLSTM cargado.
Cargando modelo GRU desde /mnt/c/Users/Raven/Desktop/ProyectoIntegrador/github/TC5035-PI-E51/Avance5/models/model_2_gru_2025-10-26_20-31-31.h5...
Modelo GRU cargado.
Cargando pipeline desde /mnt/c/Users/Raven/Desktop/ProyectoIntegrador/github/TC5035-PI-E51/Avance5/processed_data_pipeline/pipeline_procesamiento.joblib...
Pipeline cargado.
Cargando codificador desde /mnt/c/Users/Raven/Desktop/ProyectoIntegrador/github/TC5035-PI-E51/Avance5/processed_data_pipeline/label_encoder.joblib...
Codificador cargado.

--- Resumen del Modelo BiLSTM (Recurrente) ---


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 192)            │        78,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 192)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │           772 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 150, 64)        │        13,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 150, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 43,014 (168.03 KB)

 Trainable params: 43,012 (168.02 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

## 4. Procesar Videos y Realizar Predicción (Ensamble)

In [5]:
if video_files_list:
    print(f"\n--- Procesando {len(video_files_list)} Videos de la Carpeta ---")
    
    processed_videos_recurrent = pipeline.transform(video_files_list)
    
    print(f"\nDatos para BiLSTM y GRU listos. Forma: {processed_videos_recurrent.shape}")

    print("\nRealizando predicciones (Ensamble)...")
    probs_gru = model_gru.predict(processed_videos_recurrent)
    
    probs_mlp = model_mlp.predict(processed_videos_recurrent)

    W_BiLSTM = 0.25
    W_GRU = 0.75
    final_probs = (probs_gru * W_BiLSTM) + (probs_mlp * W_GRU)
    print("Predicciones combinadas.")

    print("\n--- ¡Resultados del Ensamble! ---")

    for i, video_path_str in enumerate(video_files_list):
        video_name = Path(video_path_str).name
        
        ensemble_prediction_probs = final_probs[i]
        ensemble_predicted_index = np.argmax(ensemble_prediction_probs)
        ensemble_predicted_label = encoder.inverse_transform([ensemble_predicted_index])[0]
        ensemble_confidence = np.max(ensemble_prediction_probs) * 100
        
        gru_pred_label = encoder.classes_[np.argmax(probs_gru[i])]
        gru_confidence = np.max(probs_gru[i]) * 100
        mlp_pred_label = encoder.classes_[np.argmax(probs_mlp[i])]
        mlp_confidence = np.max(probs_mlp[i]) * 100
        
        print(f"\n--- Video: {video_name} ---")
        print(f"  Predicción BiLSTM:   {gru_pred_label} ({gru_confidence:.2f}%)")
        print(f"  Predicción GRU:   {mlp_pred_label} ({mlp_confidence:.2f}%)")
        print("  ---------------------------------")
        print(f"  PREDICCIÓN FINAL (Ensamble): {ensemble_predicted_label}")
        print(f"  Confianza (Promediada):    {ensemble_confidence:.2f}%")
        print("-" * 35)
else:
    print("No se seleccionaron videos, no hay nada que predecir.")


--- Procesando 8 Videos de la Carpeta ---
Paso 1/4: Extrayendo características del video...


Extrayendo secuencia:   0%|                                                                                    | 0/8 [00:00<?, ?it/s]libEGL warning: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)



Inicializando modelo MediaPipe Hands...


libEGL warning: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1761544225.901803  176152 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1761544225.919650  176165 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/home/raven/miniconda3/envs/lsm/lib/python3.10/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.

Paso 2/4: Estandarizando longitud de secuencia...


Padding: 100%|██████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:00<00:00, 12935.40it/s]
I0000 00:00:1761544250.249819  176072 cuda_dnn.cc:529] Loaded cuDNN version 91002


Paso 3/4: Aplicando Scaler y PCA...
Características reducidas a 5 componentes.

Datos para BiLSTM y GRU listos. Forma: (8, 150, 5)

Realizando predicciones (Ensamble)...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
Predicciones combinadas.

--- ¡Resultados del Ensamble! ---

--- Video: abrir.mp4 ---
  Predicción BiLSTM:   Abrir (46.84%)
  Predicción GRU:   Borrar (68.26%)
  ---------------------------------
  PREDICCIÓN FINAL (Ensamble): Borrar
  Confianza (Promediada):    55.67%
-----------------------------------

--- Video: abrir1.mp4 ---
  Predicción BiLSTM:   Abrir (99.84%)
  Predicción GRU:   Abrir (99.98%)
  ---------------------------------
  PREDICCIÓN FINAL (Ensamble): Abrir
  Confianza (Promediada):    99.94%
-----------------------------------

--- Video: avion.mp4 ---
  Predicción BiLSTM:   Borrar (98.44%)
  Predicción GRU:   Borrar (68.88%)
  ---------------------------------
  PREDICCIÓN FINAL (Ensamble): Borrar
  Confianza (Promediada): 